In [13]:
# Install Dependencies (into the selected kernel / .venv)
%pip install -q anthropic python-dotenv

You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [14]:
import json
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
client = Anthropic()
model = "claude-sonnet-4-5-20250929"

In [15]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages,system=None,temperature=1.0,stop_sequences=None,max_tokens=1000):
    params = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text

In [16]:
def generate_dataset():
    prompt = """
    Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
    that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects
    each representing a task that requires Python, JSON, or a Regex to complete.

    Example output:
    ```json
    [
        {
            "task": "Description of task"
        }
    ]
    ```

    * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single Regex
    * Focus on tasks that do not require writing much code

    Please generate 3 objects.
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [17]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [18]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [23]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = """
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {task}
    Solution: {solution}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [26]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade  = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    } 

In [20]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results

In [27]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [28]:
print(json.dumps(results, indent=2))

[
  {
    "output": "I'll help you write a Python function to parse AWS ARN strings.\n\n```python\ndef parse_arn(arn_string):\n    \"\"\"\n    Parse an AWS ARN string and return its components as a dictionary.\n    \n    ARN Format: arn:partition:service:region:account-id:resource\n    or: arn:partition:service:region:account-id:resourcetype/resource\n    or: arn:partition:service:region:account-id:resourcetype:resource\n    \n    Args:\n        arn_string (str): The ARN string to parse\n        \n    Returns:\n        dict: A dictionary containing the ARN components\n        \n    Raises:\n        ValueError: If the ARN format is invalid\n    \"\"\"\n    if not arn_string:\n        raise ValueError(\"ARN string cannot be empty\")\n    \n    if not arn_string.startswith(\"arn:\"):\n        raise ValueError(\"ARN must start with 'arn:'\")\n    \n    # Split the ARN by colons\n    parts = arn_string.split(\":\", 5)  # Split into maximum 6 parts\n    \n    if len(parts) < 6:\n        rais